In [1]:
import os
import dash
import pandas as pd
from dash import Dash, dcc, html
from dash.dependencies import Input, Output, State, MATCH, ALL
from plotly.graph_objs import Scatter, Layout, Figure
from tvDatafeed import TvDatafeed, Interval
from fredapi import Fred
import numpy as np
import datetime

# TradingView credentials
username = 'Ousouh'
password = 'Adoboc123'
tv = TvDatafeed(username=username, password=password)

# FRED credentials
api_key = '117ba93df0ca292b31c214771432a560'
fred = Fred(api_key=api_key)



symbols = {
    # --- US Indices ---
    'S&P/TSX COMPOSITE DIVIDEND INDEX': ('TXDC', 'TSX'),
    'CA3M': ('CA03MY', 'TVC'),


    # --- US Housing Market ---
    'US: RECESSION INDICATOR': ('USREC', 'FRED')
}

data = {}
data_freq = {}
start_years = [1960,1965,1970,1975,1980,1985,1990,1995, 2000, 2005, 2010]
today = pd.Timestamp.today()
max_retries = 5

# ...existing code...
#Interval orders only for intraday variables,. 
interval_order = [
    ('1D', Interval.in_daily),
]

data = {}
start_years = [1960,1965,1970,1975,1980,1985,1990,1995, 2000, 2005, 2010]
today = pd.Timestamp.today()
max_retries = 5

for k, (symbol, exchange) in symbols.items():
    if exchange == "FRED":
        # Fetch from FRED API only
        try:
            series = fred.get_series(symbol)
            df = pd.DataFrame(series)
            df.columns = ['close']
            df.index.name = 'date'
            # Interpolate missing values after forward/backward fill
            df['close'] = df['close'].ffill().bfill().interpolate(method='linear')
            data[k] = df['close']
        except Exception as e:
            print(f"Error fetching {symbol} from FRED: {e}")
    elif exchange == "ECONOMICS":
        # Fetch from TradingView ECONOMICS (monthly)
        df = None
        for start_y in start_years:
            for attempt in range(max_retries):
                if attempt > 0:
                    print(f"Retrying {symbol}:{exchange} (attempt {attempt+1})...")
                    tv = TvDatafeed(username=username, password=password)
                start_date = pd.Timestamp(f"{start_y}-01-01")
                n_bars = min(500, ((today.year - start_date.year) * 12 + (today.month - start_date.month) + 12))
                try:
                    df = tv.get_hist(symbol=symbol, exchange=exchange, interval=Interval.in_monthly, n_bars=n_bars)
                    if df is not None and 'close' in df:
                        break
                except Exception as e:
                    print(f"Error fetching {symbol}:{exchange} from {start_y} (attempt {attempt+1}): {e}")
                    df = None
            if df is not None and 'close' in df:
                break
        if df is not None and 'close' in df:
            # Interpolate missing values after forward/backward fill
            df['close'] = df['close'].ffill().bfill().interpolate(method='linear')
            data[k] = df['close']
        else:
            print(f"Warning: No data for {k} ({symbol}:{exchange}) after {max_retries} retries per start year.")
    else:
        # Financial variables: fetch all intervals from TradingView
        for interval_name, interval_obj in interval_order:
            df = None
            for start_y in start_years:
                for attempt in range(max_retries):
                    if attempt > 0:
                        print(f"Retrying {symbol}:{exchange} {interval_name} (attempt {attempt+1})...")
                        tv = TvDatafeed(username=username, password=password)
                    start_date = pd.Timestamp(f"{start_y}-01-01")
                    if interval_name == '1M':
                        n_bars = ((today.year - start_date.year) * 12 + (today.month - start_date.month) + 12)
                    elif interval_name == '1W':
                        n_bars = ((today - start_date).days // 7) + 10
                    elif interval_name == '1D':
                        n_bars = (today - start_date).days + 10
                    else:
                        n_bars = min(10000, (today - start_date).days * 24 * 4)
                    try:
                        df = tv.get_hist(symbol=symbol, exchange=exchange, interval=interval_obj, n_bars=n_bars)
                        if df is not None and 'close' in df:
                            break
                    except Exception as e:
                        print(f"Error fetching {symbol}:{exchange} {interval_name} from {start_y} (attempt {attempt+1}): {e}")
                        df = None
                if df is not None and 'close' in df:
                    break
            if df is not None and 'close' in df:
                # Interpolate missing values after forward/backward fill
                df['close'] = df['close'].ffill().bfill().interpolate(method='linear')
                data[f"{k} [{interval_name}]"] = df['close']
            else:
                print(f"Warning: No data for {k} [{interval_name}] ({symbol}:{exchange}) after {max_retries} retries per start year.")
        else:
            # ECONOMICS/FRED: fetch only at native frequency
            freq = "D"
            if "QOQ" in k:
                freq = "Q"
            elif "YEAR-ON-YEAR" in k or "YOY" in k:
                freq = "A"
            elif "MONTH-ON-MONTH" in k or "MOM" in k:
                freq = "M"
            if exchange == "ECONOMICS" and freq == "D":
                freq = "M"

            df = None
            for start_y in start_years:
                for attempt in range(max_retries):
                    if attempt > 0:
                        print(f"Retrying {symbol}:{exchange} (attempt {attempt+1})...")
                        tv = TvDatafeed(username=username, password=password)
                    start_date = pd.Timestamp(f"{start_y}-01-01")
                    if freq == "Q":
                        n_bars = ((today.year - start_date.year) * 4 + (today.quarter - start_date.quarter) + 4)
                    elif freq == "A":
                        n_bars = ((today.year - start_date.year) * 12 + (today.month - start_date.month) + 12)
                    elif freq == "M":
                        n_bars = ((today.year - start_date.year) * 12 + (today.month - start_date.month) + 12)
                    else:
                        n_bars = (today - start_date).days + 10

                    if exchange == "TVC":
                        n_bars = min(n_bars, 5000)
                    elif exchange == "ECONOMICS":
                        n_bars = min(n_bars, 500)
                    else:
                        n_bars = min(n_bars, 11000)

                    try:
                        df = tv.get_hist(symbol=symbol, exchange=exchange, interval=Interval.in_daily, n_bars=n_bars)
                        if df is not None and 'close' in df:
                            break
                    except Exception as e:
                        print(f"Error fetching {symbol}:{exchange} from {start_y} (attempt {attempt+1}): {e}")
                        df = None
                if df is not None and 'close' in df:
                    break

            if df is not None and 'close' in df:
                # Interpolate missing values after forward/backward fill
                df['close'] = df['close'].ffill().bfill().interpolate(method='linear')
                data[k] = df['close']
            else:
                print(f"Warning: No data for {k} ({symbol}:{exchange}) after {max_retries} retries per start year.")

# --- Standardize daily data keys ---
cleaned_data = {}
for k in list(data.keys()):
    # If key ends with '[1D]', rename to base symbol (e.g., 'GOLD [1D]' -> 'GOLD')
    if k.endswith('[1D]'):
        base = k.replace(' [1D]', '')
        # Only keep the daily version, drop the base if it exists
        cleaned_data[base] = data[k]
    elif k not in cleaned_data:
        # If not a daily version, keep as is (for FRED, ECONOMICS, etc.)
        cleaned_data[k] = data[k]
data = cleaned_data

error while signin


In [2]:
# ============================
# BLOCK 2 — Add Yahoo Finance
# ============================
import pandas as pd

# --- install yfinance if missing (optional helper message) ---
try:
    import yfinance as yf
except Exception as _e:
    raise ImportError("yfinance not installed. Run: pip install yfinance") from _e

# ---- Map: label (key in `data`) -> Yahoo ticker ----
YF_MAP = {
    'DESJARDINS CANADIAN EQ INC D (NAV)': '0P0001D8SK.TO',  # your fund
    # add more if you like: 'Another Name': 'TICKER',
}

def _fetch_yf_series(ticker: str, start: str = "2010-01-01") -> pd.Series:
    """
    Download a single Yahoo ticker as a 1-D Series of closes.
    Returns tz-naive index, float dtype.
    """
    df = yf.download(ticker, start=start, interval="1d", auto_adjust=False, progress=False, threads=False)
    if df is None or df.empty or "Close" not in df:
        return pd.Series(dtype=float, name=ticker)

    # Handle possible MultiIndex from yfinance when multiple tickers slip in
    if isinstance(df.columns, pd.MultiIndex):
        s = df["Close"].iloc[:, 0].copy()
    else:
        s = df["Close"].copy()

    s.index = pd.to_datetime(s.index).tz_localize(None)
    s.name = ticker
    return s.astype(float)

# ---- Add each Yahoo series into your existing `data` dict ----
for label, ticker in YF_MAP.items():
    s = _fetch_yf_series(ticker, start="2018-01-01")  # start near fund inception
    if s.empty:
        print(f"[YF] {ticker} returned no rows.")
        continue
    # Keep the same shape as your TradingView/FRED series: a 1-D Series of closes.
    # No reindexing here — your Dash code can plot native calendars side-by-side.
    data[label] = s
    print(f"[YF] Added: {label}  (rows={len(s)}, first={s.first_valid_index().date()}, last={s.last_valid_index().date()})")

# ---- Quick sanity print: last 10 business days for Fund & TXDC ----
def _print_last_10(name: str):
    if name not in data:
        print(f"\n{name}: not found in `data`.")
        return
    s = data[name]
    if not isinstance(s, pd.Series) or s.dropna().empty:
        print(f"\n{name}: series empty.")
        return
    tail10 = s.dropna().tail(10)
    print(f"\n{name} — last 10 business days:")
    print(pd.DataFrame({name: tail10}).to_string())

# fund (from Yahoo)
_print_last_10('DESJARDINS CANADIAN EQ INC D (NAV)')

# benchmark (from your TradingView block)
_print_last_10('S&P/TSX COMPOSITE DIVIDEND INDEX')


[YF] Added: DESJARDINS CANADIAN EQ INC D (NAV)  (rows=1852, first=2018-05-11, last=2025-09-26)

DESJARDINS CANADIAN EQ INC D (NAV) — last 10 business days:
            DESJARDINS CANADIAN EQ INC D (NAV)
Date                                          
2025-09-15                              14.165
2025-09-16                              14.145
2025-09-17                              14.146
2025-09-18                              14.193
2025-09-19                              14.264
2025-09-22                              14.295
2025-09-23                              14.289
2025-09-24                              14.284
2025-09-25                              14.275
2025-09-26                              14.254

S&P/TSX COMPOSITE DIVIDEND INDEX — last 10 business days:
                     S&P/TSX COMPOSITE DIVIDEND INDEX
datetime                                             
2025-09-16 09:30:00                            225.91
2025-09-17 09:30:00                            225.92
2025-

In [3]:
# =============================================================
# BLOCK 3 — Analytics: Fund TR (perf) vs Index PO (risk) + RF CA3M
# =============================================================
import pandas as pd
import numpy as np

FUND_KEY   = 'DESJARDINS CANADIAN EQ INC D (NAV)'
BENCH_KEY  = 'S&P/TSX COMPOSITE DIVIDEND INDEX'
RF_KEY     = 'CA3M'  # 3M Canada (level in %)

FUND_LABEL  = "DJ CA Eq Inc D"
INDEX_LABEL = "S&P/TSX DIV Index"

INCEPTION_DATE  = "2018-05-11"
REBASE_YEARS    = list(range(2019, 2026))
ROLL_DAYS       = 60
TRADING_DAYS_YR = 252
MONTHS_SHARPE   = 12
MONTHS_CAPTURE  = 12
MONTHS_IR       = 12
EOM             = "ME"

# >>> CHOIX MÉTHODO
# - Perf tables (Calendar, Annual Compound, CAGR) : FUND en TR
# - Risk (vol, beta, alpha, Sharpe, captures, IR, drawdown) : FUND en Price-Only
USE_TR_FOR_PERF_TABLES = True   # oui
USE_TR_FOR_RISK        = False  # toujours False pour coller à l'industrie

def key(name: str, metric: str) -> str:
    return f"{name} | {metric}"

def _ensure_series(name):
    if name not in data:
        raise KeyError(f"`data` missing series: {name}")
    s = data[name]
    if not isinstance(s, pd.Series) or s.dropna().empty:
        raise ValueError(f"`data['{name}']` is empty.")
    s = s.copy()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    s = s.sort_index()
    return s.astype(float)

def _to_bday_series(s: pd.Series) -> pd.Series:
    idx = s.index
    if any(getattr(ts, "hour", 0) != 0 or getattr(ts, "minute", 0) != 0 for ts in idx):
        s = s.groupby(idx.normalize()).last()
    bstart, bend = s.index.min(), s.index.max()
    bidx = pd.date_range(bstart, bend, freq="B")
    return s.reindex(bidx).ffill()

# 1) Base series (B-day)
fund_close  = _to_bday_series(_ensure_series(FUND_KEY))
bench_close = _to_bday_series(_ensure_series(BENCH_KEY))
rf_level    = _to_bday_series(_ensure_series(RF_KEY))  # percent level

fund_close  = fund_close.loc[fund_close.index >= pd.Timestamp(INCEPTION_DATE)]
bench_close = bench_close.loc[bench_close.index >= fund_close.index.min()]
rf_level    = rf_level.loc[rf_level.index >= fund_close.index.min()]

bench_aligned = bench_close.reindex(fund_close.index).ffill()
px = pd.DataFrame({FUND_LABEL: fund_close, INDEX_LABEL: bench_aligned}, index=fund_close.index)

# 2) Distributions (reinvesties)
dist_rows = [
    ("2025-06-27", 0.0741), ("2025-03-28", 0.0790),
    ("2024-12-16", 0.0831), ("2024-09-27", 0.0778),
    ("2024-06-21", 0.0740), ("2024-03-22", 0.0863),
    ("2023-12-18", 0.0762), ("2023-09-22", 0.0886),
    ("2023-06-23", 0.0966), ("2023-03-24", 0.0823),
    ("2022-12-19", 0.0878), ("2022-09-23", 0.0744),
    ("2022-06-24", 0.0606), ("2022-03-25", 0.0547),
    ("2021-12-17", 0.0591), ("2021-09-24", 0.0536),
    ("2021-06-25", 0.0462), ("2021-03-26", 0.0378),
    ("2020-12-18", 0.0435), ("2020-09-25", 0.0721),
    ("2020-06-26", 0.0482), ("2020-03-27", 0.0728),
    ("2019-12-16", 0.2175), ("2019-09-27", 0.0520),
    ("2019-06-28", 0.0504), ("2019-03-29", 0.0899),
    ("2018-12-18", 0.2714), ("2018-09-28", 0.0483),
    ("2018-06-29", 0.0600),
]
def _last_trading_on_or_before(idx: pd.DatetimeIndex, date_like) -> pd.Timestamp:
    t = pd.Timestamp(date_like)
    pos = idx.searchsorted(t, side="right") - 1
    return idx[max(pos, 0)]
dist_series = pd.Series(0.0, index=px.index)
for d, amt in dist_rows:
    td = _last_trading_on_or_before(px.index, d)
    if td in dist_series.index:
        dist_series.loc[td] += float(amt)

# 3) Returns — PO (fonds & indice) + TR (fonds)
ret_d_PO = px.pct_change()
ret_m_PO = (1 + ret_d_PO).resample(EOM).prod() - 1
ret_d_TR_fund = (px[FUND_LABEL] - px[FUND_LABEL].shift(1) + dist_series) / px[FUND_LABEL].shift(1)
ret_m_TR_fund = (1 + ret_d_TR_fund).resample(EOM).prod() - 1

def _cum0_from_ret(r: pd.Series) -> pd.Series:
    return ((1 + r.dropna()).cumprod() - 1.0) * 100.0
def _dd_from_ret(r: pd.Series) -> pd.Series:
    w = (1 + r.dropna()).cumprod()
    return (w / w.cummax() - 1.0) * 100.0

cum0_TR_fund = _cum0_from_ret(ret_d_TR_fund)
cum0_PO = pd.DataFrame({
    FUND_LABEL:  _cum0_from_ret(ret_d_PO[FUND_LABEL]),
    INDEX_LABEL: _cum0_from_ret(ret_d_PO[INDEX_LABEL]),
})
dd_pct_PO = pd.DataFrame({
    FUND_LABEL:  _dd_from_ret(ret_d_PO[FUND_LABEL]),
    INDEX_LABEL: _dd_from_ret(ret_d_PO[INDEX_LABEL]),
})

# >>> SÉRIES DE RÉFÉRENCE POUR CHAQUE BLOC
# Rolling beta/alpha/R2 (RISK) : price-only daily
fund_ret_d_for_roll = ret_d_PO[FUND_LABEL]
bench_ret_d         = ret_d_PO[INDEX_LABEL]
# Sharpe / captures / IR (RISK) : price-only monthly
fund_ret_m_for_risk = ret_m_PO[FUND_LABEL]
bench_ret_m_for_risk= ret_m_PO[INDEX_LABEL]
# Tables de performance : TR pour le fonds, PO pour l’indice
fund_ret_m_for_perf = ret_m_TR_fund if USE_TR_FOR_PERF_TABLES else ret_m_PO[FUND_LABEL]
bench_ret_m_for_perf= ret_m_PO[INDEX_LABEL]

# 4) Rolling beta/alpha/R2 (toujours PO)
def _roll_beta_alpha_r2(asset: pd.Series, bench: pd.Series, win: int):
    df = pd.concat([asset, bench], axis=1, join="inner").dropna()
    df.columns = ["a","m"]
    mean_a = df["a"].rolling(win).mean()
    mean_m = df["m"].rolling(win).mean()
    cov    = df["a"].rolling(win).cov(df["m"])
    var_m  = df["m"].rolling(win).var()
    beta   = cov / var_m
    alpha_d = mean_a - beta * mean_m
    alpha_ann_pct = ((1 + alpha_d) ** TRADING_DAYS_YR - 1.0) * 100.0
    corr   = df["a"].rolling(win).corr(df["m"])
    r2     = corr * corr
    out = pd.DataFrame({"beta": beta, "alpha_ann_pct": alpha_ann_pct, "r2": r2})
    return out.reindex(px.index)

roll_f = _roll_beta_alpha_r2(fund_ret_d_for_roll, bench_ret_d, ROLL_DAYS)
roll_i = pd.DataFrame(index=px.index, data={"beta": 1.0, "alpha_ann_pct": 0.0, "r2": 1.0})

# 5) RF dynamique (CA3M) + Sharpe / Captures / IR (tous PO)
rf_annual_decimal = (rf_level / 100.0)
rf_m_series = (1 + rf_annual_decimal).resample(EOM).mean().pow(1/12) - 1

def _monthly_to_daily_interp(m: pd.Series, daily_index: pd.DatetimeIndex) -> pd.Series:
    if m.dropna().empty:
        return pd.Series(dtype=float, index=daily_index)
    s = m.copy()
    s.index = s.index.to_period("M").to_timestamp("M")
    full_daily = pd.date_range(daily_index.min(), daily_index.max(), freq="D")
    d = s.reindex(full_daily).interpolate("time").ffill()
    return d.reindex(daily_index, method="ffill")

def sharpe_rolling_12m_monthly(asset_rm: pd.Series, rf_m: pd.Series) -> pd.Series:
    ex = pd.concat([asset_rm, rf_m], axis=1, join="inner").dropna()
    ex.columns = ["rm","rfm"]; ex["ex"] = ex["rm"] - ex["rfm"]
    def _calc(x):
        s = pd.Series(x); sd = s.std(ddof=1)
        return (s.mean() / sd) * np.sqrt(12) if sd and sd > 0 else np.nan
    return ex["ex"].rolling(MONTHS_SHARPE).apply(_calc, raw=False)

def capture_rolling_12m(asset_rm: pd.Series, bench_rm: pd.Series, window: int = 12):
    df = pd.concat([asset_rm, bench_rm], axis=1, join="inner").dropna()
    df.columns = ["a","m"]
    up   = pd.Series(index=df.index, dtype=float)
    down = pd.Series(index=df.index, dtype=float)
    for i in range(window-1, len(df)):
        w = df.iloc[i-window+1:i+1]
        up.iloc[i]   = (w[w["m"]>0]["a"].sum() / w[w["m"]>0]["m"].sum()) if (w["m"]>0).any() else np.nan
        down.iloc[i] = (w[w["m"]<0]["a"].sum() / w[w["m"]<0]["m"].sum()) if (w["m"]<0).any() else np.nan
    return up, down

def info_ratio_rolling_12m_monthly(asset_rm: pd.Series, bench_rm: pd.Series, window: int = 12) -> pd.Series:
    df = pd.concat([asset_rm, bench_rm], axis=1, join="inner").dropna()
    active = df.iloc[:,0] - df.iloc[:,1]
    def _ir(x):
        s = pd.Series(x); sd = s.std(ddof=1)
        return (s.mean() / sd) * np.sqrt(12) if sd and sd > 0 else np.nan
    return active.rolling(window).apply(_ir, raw=False)

# -> RISK on PO
sh_m_f = sharpe_rolling_12m_monthly(fund_ret_m_for_risk, rf_m_series)
sh_m_i = sharpe_rolling_12m_monthly(bench_ret_m_for_risk, rf_m_series)
sh_12m_f = _monthly_to_daily_interp(sh_m_f, px.index)
sh_12m_i = _monthly_to_daily_interp(sh_m_i, px.index)

up_m_f, down_m_f = capture_rolling_12m(fund_ret_m_for_risk, bench_ret_m_for_risk, MONTHS_CAPTURE)
up_m_i, down_m_i = capture_rolling_12m(bench_ret_m_for_risk, bench_ret_m_for_risk, MONTHS_CAPTURE)
up_12m_f   = _monthly_to_daily_interp(up_m_f,   px.index)
down_12m_f = _monthly_to_daily_interp(down_m_f, px.index)
up_12m_i   = _monthly_to_daily_interp(up_m_i,   px.index)
down_12m_i = _monthly_to_daily_interp(down_m_i, px.index)

ir_m_f   = info_ratio_rolling_12m_monthly(fund_ret_m_for_risk, bench_ret_m_for_risk, MONTHS_IR)
ir_12m_f = _monthly_to_daily_interp(ir_m_f, px.index)

# 6) Rebases (inchangé)
def _first_trading_on_or_after(idx: pd.DatetimeIndex, date_like: str) -> pd.Timestamp:
    t = pd.Timestamp(date_like); pos = idx.searchsorted(t, side="left")
    return idx[min(max(pos, 0), len(idx)-1)]
def _rebased_year_from_daily_ret(ret_daily: pd.Series, idx: pd.DatetimeIndex, year: int) -> pd.Series:
    start_anchor = _last_trading_on_or_before(idx, f"{year-1}-12-31")
    start_date   = _first_trading_on_or_after(idx, f"{year}-01-01")
    end_date     = _last_trading_on_or_before(idx, f"{year}-12-31")
    yret = ret_daily.loc[(ret_daily.index>=start_date)&(ret_daily.index<=end_date)]
    cum0_y = ((1 + yret).cumprod() - 1.0) * 100.0
    out = pd.concat([pd.Series([0.0], index=[start_anchor]), cum0_y]).sort_index()
    return out[~out.index.duplicated(keep="first")]

rebased = {}
for y in REBASE_YEARS:
    rebased[f"{FUND_LABEL} | Reb {y} Cum0 % (PO)"]  = _rebased_year_from_daily_ret(ret_d_PO[FUND_LABEL].dropna(), px.index, y)
    rebased[f"{FUND_LABEL} | Reb {y} Cum0 % (TR)"]  = _rebased_year_from_daily_ret(ret_d_TR_fund.dropna(),          px.index, y)
    rebased[f"{INDEX_LABEL} | Reb {y} Cum0 %"]      = _rebased_year_from_daily_ret(ret_d_PO[INDEX_LABEL].dropna(),   px.index, y)

# 7) Emission dans `data`
# FUND (Price-only pour risk series / TR pour cum0 TR)
data[key(FUND_LABEL,  "Price")]          = px[FUND_LABEL].dropna()
data[key(FUND_LABEL,  "Ret D (PO)")]     = ret_d_PO[FUND_LABEL].dropna()
data[key(FUND_LABEL,  "Ret D (TR)")]     = ret_d_TR_fund.dropna()
data[key(FUND_LABEL,  "Cum0 % (PO)")]    = cum0_PO[FUND_LABEL].dropna()
data[key(FUND_LABEL,  "Cum0 % (TR)")]    = cum0_TR_fund.dropna()
data[key(FUND_LABEL,  "DD % (PO)")]      = dd_pct_PO[FUND_LABEL].dropna()
data[key(FUND_LABEL,  "Vol 60j % (PO)")] = (fund_ret_d_for_roll.rolling(ROLL_DAYS).std(ddof=1) * np.sqrt(TRADING_DAYS_YR) * 100.0).dropna()
data[key(FUND_LABEL,  "Beta 60j (PO)")]  = roll_f["beta"].dropna()
data[key(FUND_LABEL,  "Alpha 60j % (PO)")] = roll_f["alpha_ann_pct"].dropna()
data[key(FUND_LABEL,  "R2 60j (PO)")]    = roll_f["r2"].dropna()
data[key(FUND_LABEL,  "Sharpe 12m (PO)")]   = sh_12m_f.dropna()
data[key(FUND_LABEL,  "UpCap 12m (PO)")]    = up_12m_f.dropna()
data[key(FUND_LABEL,  "DownCap 12m (PO)")]  = down_12m_f.dropna()
data[key(FUND_LABEL,  "InfoRatio 12m (PO)")] = ir_12m_f.dropna()

# INDEX (toujours PO)
data[key(INDEX_LABEL, "Price")]          = px[INDEX_LABEL].dropna()
data[key(INDEX_LABEL, "Ret D")]          = ret_d_PO[INDEX_LABEL].dropna()
data[key(INDEX_LABEL, "Cum0 %")]         = cum0_PO[INDEX_LABEL].dropna()
data[key(INDEX_LABEL, "DD %")]           = dd_pct_PO[INDEX_LABEL].dropna()
data[key(INDEX_LABEL, "Vol 60j %")]      = (bench_ret_d.rolling(ROLL_DAYS).std(ddof=1) * np.sqrt(TRADING_DAYS_YR) * 100.0).dropna()
data[key(INDEX_LABEL, "Beta 60j")]       = roll_i["beta"].dropna()
data[key(INDEX_LABEL, "Alpha 60j %")]    = roll_i["alpha_ann_pct"].dropna()
data[key(INDEX_LABEL, "R2 60j")]         = roll_i["r2"].dropna()
data[key(INDEX_LABEL, "Sharpe 12m")]     = sh_12m_i.dropna()
data[key(INDEX_LABEL, "UpCap 12m")]      = up_12m_i.dropna()
data[key(INDEX_LABEL, "DownCap 12m")]    = down_12m_i.dropna()

# Spread TR (fonds) vs PO (indice)
_overlap = pd.concat([cum0_TR_fund, cum0_PO[INDEX_LABEL]], axis=1, join="inner").dropna()
if not _overlap.empty:
    data[f"{FUND_LABEL} (TR) - {INDEX_LABEL} | Cum0 Spread pts"] = _overlap.iloc[:,0] - _overlap.iloc[:,1]

data.update(rebased)

# 8) Tables — PERF en TR (fonds), INDICE en PO (standard fiche)
def _cal_from_monthly(rm: pd.Series) -> pd.Series:
    if rm.dropna().empty: return pd.Series(dtype=float)
    return rm.groupby(rm.index.year).apply(lambda s: (1 + s).prod() - 1) * 100.0

ret_m_f = fund_ret_m_for_perf
ret_m_i = bench_ret_m_for_perf
ret_m_i_aligned = ret_m_i.loc[ret_m_f.index.min():].copy()

def _ytd_from_monthly(rm: pd.Series) -> float:
    if rm.dropna().empty: return np.nan
    y = rm.index.max().year
    s = rm[rm.index.year == y]
    return ((1 + s).prod() - 1) * 100.0 if not s.dropna().empty else np.nan

cal_f = _cal_from_monthly(ret_m_f)
cal_i = _cal_from_monthly(ret_m_i_aligned)

asof = px.index.max().date() if not px.empty else None
ytd_f = _ytd_from_monthly(ret_m_f)
ytd_i = _ytd_from_monthly(ret_m_i_aligned)

years = sorted(set(cal_f.index.tolist() + cal_i.index.tolist()))
calendar_table = pd.DataFrame(index=["D Class","Index"],
                              columns=["YTD"] + [str(y) for y in years],
                              dtype=float)
calendar_table.loc["D Class","YTD"] = round(ytd_f, 2) if pd.notna(ytd_f) else np.nan
calendar_table.loc["Index","YTD"]   = round(ytd_i, 2) if pd.notna(ytd_i) else np.nan
for y in years:
    calendar_table.loc["D Class", str(y)] = round(cal_f.get(y, np.nan), 2)
    calendar_table.loc["Index",  str(y)]  = round(cal_i.get(y, np.nan), 2)

def _period_ret_m(rm: pd.Series, months: int) -> float:
    s = rm.dropna().iloc[-months:]
    return ((1 + s).prod() - 1) * 100.0 if len(s) == months else np.nan
def _ann_from_window(rm: pd.Series, months: int) -> float:
    s = rm.dropna().iloc[-months:]; 
    if len(s) < months: return np.nan
    growth = (1 + s).prod()
    return (growth ** (12 / months) - 1) * 100.0
def _since_inception_m(rm: pd.Series) -> float:
    s = rm.dropna(); 
    return ((1 + s).prod() - 1) * 100.0 if not s.empty else np.nan
def _cagr_from_cum0(c0: pd.Series) -> float:
    s = (c0.dropna()/100.0 + 1.0)
    if s.empty: return np.nan
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    return ((s.iloc[-1] / s.iloc[0]) ** (1/yrs) - 1) * 100.0 if yrs > 0 else np.nan

annual_cols = ["1 Mth","3 Mth","6 Mth","1 Yr","3 Yr (ann)","5 Yr (ann)","10 Yr (ann)","Since Inception","CAGR (SI)"]
annual_comp_table = pd.DataFrame(index=["D Class"], columns=annual_cols, dtype=float)
rmF = ret_m_f
annual_comp_table.loc["D Class","1 Mth"]           = round(_period_ret_m(rmF, 1), 2)
annual_comp_table.loc["D Class","3 Mth"]           = round(_period_ret_m(rmF, 3), 2)
annual_comp_table.loc["D Class","6 Mth"]           = round(_period_ret_m(rmF, 6), 2)
annual_comp_table.loc["D Class","1 Yr"]            = round(_period_ret_m(rmF, 12), 2)
annual_comp_table.loc["D Class","3 Yr (ann)"]      = round(_ann_from_window(rmF, 36), 2)
annual_comp_table.loc["D Class","5 Yr (ann)"]      = round(_ann_from_window(rmF, 60), 2)
annual_comp_table.loc["D Class","10 Yr (ann)"]     = np.nan
cum_for_cagr = cum0_TR_fund  # toujours TR pour le fonds
annual_comp_table.loc["D Class","Since Inception"] = round(_since_inception_m(rmF), 2)
annual_comp_table.loc["D Class","CAGR (SI)"]       = round(_cagr_from_cum0(cum_for_cagr), 2)

# Risk table (3 ans / 5 ans) — TOUJOURS PO
def risk_measures(asset_rm_PO: pd.Series, bench_rm_PO: pd.Series, rf_m: pd.Series, months: int):
    df = pd.concat([asset_rm_PO, bench_rm_PO, rf_m], axis=1, join="inner").dropna().iloc[-months:]
    df.columns = ["a","m","rfm"]
    if df.shape[0] < max(24, months//2):
        return {k: np.nan for k in ["Stdev_Ann_%","Beta","Alpha_Ann_%","R2","Sharpe","UpCap","DownCap","InfoRatio","MaxDD_%"]}
    stdev_ann = df["a"].std(ddof=1) * np.sqrt(12) * 100.0
    beta = df["a"].cov(df["m"]) / df["m"].var() if df["m"].var() else np.nan
    alpha_m = (df["a"] - beta * df["m"]).mean() if pd.notna(beta) else np.nan
    alpha_ann = ((1 + alpha_m)**12 - 1) * 100.0 if pd.notna(alpha_m) else np.nan
    r2 = (df["a"].corr(df["m"]) ** 2) if df["a"].std(ddof=1) and df["m"].std(ddof=1) else np.nan
    ex = df["a"] - df["rfm"]
    sharpe = (ex.mean() / ex.std(ddof=1)) * np.sqrt(12) if ex.std(ddof=1) > 0 else np.nan
    up   = df[df["m"] > 0];   down = df[df["m"] < 0]
    upcap   = (up["a"].sum()   / up["m"].sum())   if not up.empty   and up["m"].sum()   != 0 else np.nan
    downcap = (down["a"].sum() / down["m"].sum()) if not down.empty and down["m"].sum() != 0 else np.nan
    active = df["a"] - df["m"]
    ir = (active.mean() / active.std(ddof=1)) * np.sqrt(12) if active.std(ddof=1) and active.std(ddof=1) > 0 else np.nan

    # Max Drawdown en PO (utilise ret_m_PO -> approx mensuelle)
    cum = (1 + df["a"]).cumprod(); dd = cum / cum.cummax() - 1
    maxdd = dd.min() * 100.0
    return {
        "Stdev_Ann_%": round(stdev_ann, 2),
        "Beta": round(beta, 2) if pd.notna(beta) else np.nan,
        "Alpha_Ann_%": round(alpha_ann, 2) if pd.notna(alpha_ann) else np.nan,
        "R2": round(r2, 2) if pd.notna(r2) else np.nan,
        "Sharpe": round(sharpe, 2) if pd.notna(sharpe) else np.nan,
        "UpCap": round(upcap, 2) if pd.notna(upcap) else np.nan,
        "DownCap": round(downcap, 2) if pd.notna(downcap) else np.nan,
        "InfoRatio": round(ir, 2) if pd.notna(ir) else np.nan,
        "MaxDD_%": round(maxdd, 2) if pd.notna(maxdd) else np.nan,
    }

risk_3y = risk_measures(fund_ret_m_for_risk, bench_ret_m_for_risk, rf_m_series, months=36)
risk_5y = risk_measures(fund_ret_m_for_risk, bench_ret_m_for_risk, rf_m_series, months=60)
risk_table = pd.DataFrame.from_dict({"3 Years": risk_3y, "5 Years": risk_5y}, orient="index")

# 9) Overview
asof  = px.index.max().date() if not px.empty else None
nav   = float(px[FUND_LABEL].dropna().iloc[-1]) if FUND_LABEL in px.columns else np.nan
prev  = float(px[FUND_LABEL].dropna().iloc[-2]) if FUND_LABEL in px.columns and px[FUND_LABEL].dropna().shape[0] >= 2 else np.nan
chg   = nav - prev if pd.notna(nav) and pd.notna(prev) else np.nan
chg_pct = (chg / prev * 100.0) if pd.notna(chg) and prev else np.nan
overview = pd.DataFrame([{
    "Name": FUND_LABEL,
    "NAV": round(nav, 4) if pd.notna(nav) else np.nan,
    "As_Of": str(asof),
    "Daily_Change_$": round(chg, 4) if pd.notna(chg) else np.nan,
    "Daily_Change_%": round(chg_pct, 3) if pd.notna(chg_pct) else np.nan
}]).set_index("Name")

tables = {
    "overview": overview,
    "calendar_title": f"Calendar Year Returns (%) — D Class as at {asof} — (Total Return)",
    "calendar": calendar_table,
    "annual_comp_title": f"Annual Compound Returns (%) — D Class as at {asof} (TR)",
    "annual_comp": annual_comp_table,
    "risk_measures": risk_table
}

print(f"[OK] Analytics added. data now has {len(data)} series.")
print("Sample keys:", list(data.keys())[:10])
print("\n=== OVERVIEW ===\n", overview)
print("\n=== CALENDAR ===\n", tables['calendar_title'], "\n", calendar_table)
print("\n=== ANNUAL COMPOUND ===\n", tables['annual_comp_title'], "\n", annual_comp_table)
print("\n=== RISK (3Y/5Y) ===\n", risk_table)


[OK] Analytics added. data now has 51 series.
Sample keys: ['S&P/TSX COMPOSITE DIVIDEND INDEX', 'CA3M', 'US: RECESSION INDICATOR', 'DESJARDINS CANADIAN EQ INC D (NAV)', 'DJ CA Eq Inc D | Price', 'DJ CA Eq Inc D | Ret D (PO)', 'DJ CA Eq Inc D | Ret D (TR)', 'DJ CA Eq Inc D | Cum0 % (PO)', 'DJ CA Eq Inc D | Cum0 % (TR)', 'DJ CA Eq Inc D | DD % (PO)']

=== OVERVIEW ===
                    NAV       As_Of  Daily_Change_$  Daily_Change_%
Name                                                              
DJ CA Eq Inc D  14.254  2025-09-26          -0.021          -0.147

=== CALENDAR ===
 Calendar Year Returns (%) — D Class as at 2025-09-26 — (Total Return) 
            YTD  2018   2019  2020   2021  2022  2023   2024   2025
D Class  13.44 -7.38  17.69 -9.17  24.90  0.42  8.62  16.97  13.44
Index    18.26 -9.88  17.67 -2.63  23.86 -3.33  5.74  15.85  18.26

=== ANNUAL COMPOUND ===
 Annual Compound Returns (%) — D Class as at 2025-09-26 (TR) 
          1 Mth  3 Mth  6 Mth   1 Yr  3 Yr (ann)  

In [4]:
from dash import ctx
import os
import dash
import pandas as pd
from dash import Dash, dcc, html
from dash.dependencies import Input, Output, State, MATCH, ALL
from plotly.graph_objs import Scatter, Layout, Figure
from tvDatafeed import TvDatafeed, Interval
from fredapi import Fred
import numpy as np
import datetime

app = Dash(__name__, suppress_callback_exceptions=True)
server = app.server

def panel_controls(panel_id, data_keys):
    return html.Div([
        dcc.Dropdown(
            id={'type': 'series-selector', 'index': panel_id},
            options=[{'label': k, 'value': k} for k in data_keys],
            multi=True,
            placeholder="Series",
            style={
                'marginBottom': '6px',
                'fontSize': '8px',
                'background': 'white',
                'color': 'black',
                'width': '180px',
                'minWidth': '50px',
                'maxWidth': '150px'
            }
        ),
        html.Div(id={'type': 'series-options-container', 'index': panel_id}),
        dcc.Input(
            id={'type': 'sr-lines', 'index': panel_id},
            type='text',
            placeholder='Support/Resistance (comma sep)',
            style={'width': '100%', 'marginBottom': '6px', 'fontSize': '8px', 'background': 'white', 'color': 'black'}
        ),
        dcc.Input(
            id={'type': 'xrange-start', 'index': panel_id},
            type='text',
            placeholder='YYYY-MM-DD (start date)',
            style={'width': '100%', 'marginBottom': '6px', 'fontSize': '8px', 'background': 'white', 'color': 'black'}
        ),
        dcc.Input(
            id={'type': 'xrange-end', 'index': panel_id},
            type='text',
            placeholder='YYYY-MM-DD (end date)',
            style={'width': '100%', 'marginBottom': '6px', 'fontSize': '8px', 'background': 'white', 'color': 'black'}
        ),
        html.Div([
            html.Div([
                dcc.Input(
                    id={'type': 'yls-min', 'index': panel_id},
                    type='number',
                    placeholder='LS min',
                    style={
                        'width': '45px', 'height': '18px', 'fontSize': '8px',
                        'background': 'white', 'color': 'black', 'marginRight': '3px',
                        'padding': '0 2px', 'border': '1px solid #eee', 'borderRadius': '3px'
                    }
                ),
                dcc.Input(
                    id={'type': 'yls-max', 'index': panel_id},
                    type='number',
                    placeholder='LS max',
                    style={
                        'width': '45px', 'height': '18px', 'fontSize': '8px',
                        'background': 'white', 'color': 'black',
                        'padding': '0 2px', 'border': '1px solid #eee', 'borderRadius': '3px'
                    }
                ),
            ], style={'display': 'inline-block', 'marginRight': '8px'}),
            html.Div([
                dcc.Input(
                    id={'type': 'yrs-min', 'index': panel_id},
                    type='number',
                    placeholder='RS min',
                    style={
                        'width': '45px', 'height': '18px', 'fontSize': '8px',
                        'background': 'white', 'color': 'black', 'marginRight': '3px',
                        'padding': '0 2px', 'border': '1px solid #eee', 'borderRadius': '3px'
                    }
                ),
                dcc.Input(
                    id={'type': 'yrs-max', 'index': panel_id},
                    type='number',
                    placeholder='RS max',
                    style={
                        'width': '45px', 'height': '18px', 'fontSize': '8px',
                        'background': 'white', 'color': 'black',
                        'padding': '0 2px', 'border': '1px solid #eee', 'borderRadius': '3px'
                    }
                ),
            ], style={'display': 'inline-block'}),
        ], style={'marginBottom': '6px', 'marginTop': '2px'}),
        dcc.RadioItems(
            id={'type': 'axis-selector', 'index': panel_id},
            options=[
                {'label': 'Single', 'value': 'single'},
                {'label': 'Dual', 'value': 'dual'}
            ],
            value='single',
            labelStyle={'display': 'inline-block', 'marginRight': '8px', 'fontSize': '8px', 'color': 'black'}
        ),
        html.Div([
            dcc.Input(id={'type': 'left-unit', 'index': panel_id}, type='text', value='', placeholder='LS', style={'width': '40px', 'marginRight': '6px', 'fontSize': '8px', 'background': 'white', 'color': 'black'}),
            html.Button("↕", id={'type': 'invert-ls', 'index': panel_id}, n_clicks=0, title="Invert LS", style={'fontSize': '8px', 'marginRight': '8px', 'background': 'white', 'color': 'black', 'border': '1px solid #eee', 'borderRadius': '3px', 'padding': '2px 6px'}),
            dcc.Input(id={'type': 'right-unit', 'index': panel_id}, type='text', value='', placeholder='RS', style={'width': '40px', 'marginRight': '6px', 'fontSize': '8px', 'background': 'white', 'color': 'black'}),
            html.Button("↕", id={'type': 'invert-rs', 'index': panel_id}, n_clicks=0, title="Invert RS", style={'fontSize': '8px', 'background': 'white', 'color': 'black', 'border': '1px solid #eee', 'borderRadius': '3px', 'padding': '2px 6px'}),
        ], style={'marginBottom': '6px'}),
        dcc.Checklist(
            id={'type': 'recession-toggle', 'index': panel_id},
            options=[{'label': 'Show Recession Shading', 'value': 'show'}],
            value=[],
            style={'fontSize': '8px', 'marginBottom': '6px', 'color': 'black'}
        ),
        dcc.Textarea(id={'type': 'footnotes', 'index': panel_id}, placeholder='Footnotes', style={'width': '100%', 'height': '28px', 'fontSize': '8px', 'background': 'white', 'color': 'black', 'border': '1px solid #eee', 'marginBottom': '4px'}),
        html.Div([
            dcc.Input(id={'type': 'chart-width', 'index': panel_id}, type='number', value=600, min=200, max=2000, step=10, style={'width': '60px', 'fontSize': '8px', 'marginRight': '8px'}),
            dcc.Input(id={'type': 'chart-height', 'index': panel_id}, type='number', value=350, min=100, max=1200, step=10, style={'width': '60px', 'fontSize': '8px'}),
        ], style={'marginBottom': '6px'}),
        html.Div([
            html.Button("⬇ PNG", id={'type': 'export-png', 'index': panel_id}, n_clicks=0, title="Export PNG", style={'fontSize': '8px', 'background': 'white', 'color': '#bdbdbd', 'border': 'none', 'padding': '0 8px', 'cursor': 'pointer', 'marginRight': '8px'}),
            html.Button("⬇ SVG", id={'type': 'export-svg', 'index': panel_id}, n_clicks=0, title="Export SVG", style={'fontSize': '8px', 'background': 'white', 'color': '#bdbdbd', 'border': 'none', 'padding': '0 8px', 'cursor': 'pointer'}),
        ], style={'marginBottom': '6px'}),
    ], style={'marginBottom': '6px', 'padding': '0'})  # removed background

@app.callback(
    Output({'type': 'series-options-container', 'index': MATCH}, 'children'),
    Input({'type': 'series-selector', 'index': MATCH}, 'value'),
    prevent_initial_call=True
)
def render_series_options(selected_series):
    if not selected_series:
        return []
    panel_id = ctx.triggered_id['index'] if hasattr(ctx, 'triggered_id') and ctx.triggered_id else 0
    return [
        html.Div([
            html.Span(s, style={'fontSize': '8px', 'marginRight': '6px'}),
            dcc.RadioItems(
                id={'type': 'series-axis', 'index': f"{panel_id}-{i}"},
                options=[
                    {'label': 'LS', 'value': 'y'},
                    {'label': 'RS', 'value': 'y2'},
                    {'label': 'Column', 'value': 'column'}
                ],
                value='y' if i < 2 else 'y2',
                labelStyle={'display': 'inline-block', 'fontSize': '8px', 'marginRight': '8px'}
            ),
            dcc.RadioItems(
                id={'type': 'series-chart-type', 'index': f"{panel_id}-{i}"},
                options=[
                    {'label': 'Line', 'value': 'line'},
                    {'label': 'Column', 'value': 'column'}
                ],
                value='line',
                labelStyle={'display': 'inline-block', 'fontSize': '8px', 'marginRight': '8px'}
            )
        ], style={'marginBottom': '2px'})
        for i, s in enumerate(selected_series)
    ]

def panel_row(panel_id, data_keys, n_panels):
    return html.Div([
        html.Div([
            panel_controls(panel_id, sorted(data_keys)),
            html.Button(
                "🗑",
                id={'type': 'remove-panel', 'index': panel_id},
                n_clicks=0,
                title="Remove Panel",
                style={
                    'fontSize': '16px',
                    'background': 'white',
                    'color': '#bdbdbd',
                    'border': 'none',
                    'marginTop': '8px',
                    'marginLeft': '0',
                    'padding': '0 4px',
                    'cursor': 'pointer',
                    'display': 'block' if n_panels > 1 else 'none'
                }
            )
        ], style={
            'width': '140px',
            'display': 'inline-block',
            'verticalAlign': 'top',
            'paddingRight': '0'
        }),
        dcc.Store(id={'type': 'xrange-store', 'index': panel_id}),
        html.Div([
            dcc.Graph(
                id={'type': 'chart', 'index': panel_id},
                config={'displayModeBar': False},
                style={
                    'height': 'auto',
                    'width': 'auto',
                    'border': '1px solid #eee',
                    'borderRadius': '4px'
                }
            )
        ], style={
            'width': 'auto',
            'display': 'inline-block',
            'verticalAlign': 'top',
            'marginLeft': '12px'
        })
    ],
    style={
        'display': 'flex',
        'flexDirection': 'row',
        'alignItems': 'flex-start',
        'marginBottom': '40px'
    })

app.layout = html.Div(
    style={
        'width': '600px',
        'margin': 'auto',
        'fontFamily': 'Segoe UI, Arial',
        'color': 'black',
        'paddingTop': '18px'
    },
    children=[
        html.Div([
            html.Div("GENESIS RESEARCH", style={
                'fontWeight': 'bold',
                'fontSize': '15px',
                'color': 'black',
                'letterSpacing': '2px',
                'marginBottom': '0'
            }),
            html.Div("Macro Chart", style={
                'fontSize': '10px',
                'color': '#888',
                'marginBottom': '10px'
            }),
        ], style={'textAlign': 'center', 'paddingBottom': '6px'}),
        html.Div([
            html.Button("＋", id='add-panel', n_clicks=1, title="Add Panel",
                        style={'fontSize': '8px', 'background': 'white', 'color': '#bdbdbd', 'border': 'none', 'padding': '0 8px', 'cursor': 'pointer', 'marginRight': '8px'}),
        ], style={'textAlign': 'left', 'marginBottom': '8px', 'marginLeft': '8px'}),
        dcc.Store(id='panel-ids-store', data=[0]),
        html.Div(id='panels-container')
    ]
)

@app.callback(
    Output('panel-ids-store', 'data'),
    Output('panels-container', 'children'),
    Input('add-panel', 'n_clicks'),
    Input({'type': 'remove-panel', 'index': ALL}, 'n_clicks'),
    State('panel-ids-store', 'data'),
    State('panels-container', 'children'),
)
def display_panels(n_add, n_remove, panel_ids, children):
    ctx_trigger = dash.callback_context.triggered[0]['prop_id']
    if panel_ids is None or len(panel_ids) == 0:
        panel_ids = [0]
    if 'remove-panel' in ctx_trigger:
        idx = int(ctx_trigger.split('.')[0].split('index\\":')[1].split('}')[0])
        panel_ids = [i for i in panel_ids if i != idx]
        if not panel_ids:
            panel_ids = [0]
    elif 'add-panel' in ctx_trigger:
        # Only add a new panel, keep others unchanged
        panel_ids = panel_ids + [max(panel_ids) + 1 if panel_ids else 0]
    n_panels = len(panel_ids)
    return panel_ids, [panel_row(i, sorted(data.keys()), n_panels) for i in panel_ids]

@app.callback(
    Output({'type': 'chart', 'index': MATCH}, 'figure'),
    Output({'type': 'xrange-store', 'index': MATCH}, 'data'),
    Input({'type': 'series-selector', 'index': MATCH}, 'value'),
    Input({'type': 'series-axis', 'index': ALL}, 'value'),
    State({'type': 'series-axis', 'index': ALL}, 'id'),
    Input({'type': 'series-chart-type', 'index': ALL}, 'value'),
    State({'type': 'series-chart-type', 'index': ALL}, 'id'),
    Input({'type': 'axis-selector', 'index': MATCH}, 'value'),
    Input({'type': 'left-unit', 'index': MATCH}, 'value'),
    Input({'type': 'right-unit', 'index': MATCH}, 'value'),
    Input({'type': 'footnotes', 'index': MATCH}, 'value'),
    Input({'type': 'invert-ls', 'index': MATCH}, 'n_clicks'),
    Input({'type': 'invert-rs', 'index': MATCH}, 'n_clicks'),
    Input({'type': 'sr-lines', 'index': MATCH}, 'value'),
    Input({'type': 'xrange-start', 'index': MATCH}, 'value'),
    Input({'type': 'xrange-end', 'index': MATCH}, 'value'),
    Input({'type': 'yls-min', 'index': MATCH}, 'value'),
    Input({'type': 'yls-max', 'index': MATCH}, 'value'),
    Input({'type': 'yrs-min', 'index': MATCH}, 'value'),
    Input({'type': 'yrs-max', 'index': MATCH}, 'value'),
    Input({'type': 'chart', 'index': MATCH}, 'relayoutData'),
    State({'type': 'xrange-store', 'index': MATCH}, 'data'),
    Input({'type': 'recession-toggle', 'index': MATCH}, 'value'),
    Input({'type': 'chart-width', 'index': MATCH}, 'value'),
    Input({'type': 'chart-height', 'index': MATCH}, 'value'),
    prevent_initial_call=True
)
def update_chart(selected_series, axis_assignments, axis_ids, chart_types, chart_type_ids, axis_mode, left_unit, right_unit, footnote, invert_ls, invert_rs, sr_lines,
                 xrange_start, xrange_end, yls_min, yls_max, yrs_min, yrs_max, relayout_data, stored_range, recession_toggle,
                 chart_width, chart_height):
    fig = Figure()
    if not selected_series:
        return fig, stored_range

    panel_id = ctx.triggered_id['index'] if hasattr(ctx, 'triggered_id') and ctx.triggered_id else 0

    axis_map = {}
    chart_type_map = {}
    for val, iddict in zip(axis_assignments, axis_ids):
        idx = iddict['index']
        if isinstance(idx, str) and idx.startswith(f"{panel_id}-"):
            series_idx = int(idx.split('-')[1])
            axis_map[series_idx] = val
    for val, iddict in zip(chart_types, chart_type_ids):
        idx = iddict['index']
        if isinstance(idx, str) and idx.startswith(f"{panel_id}-"):
            series_idx = int(idx.split('-')[1])
            chart_type_map[series_idx] = val

    invert_left = invert_ls % 2 == 1
    invert_right = invert_rs % 2 == 1
    color_order = [
        '#455a64',  # blue-grey
        '#0277bd',  # vivid blue
        '#ffb300',  # amber
        '#c2185b',  # magenta
        '#43a047',  # green
        '#f4511e',  # orange-red
        '#7e57c2'   # purple
    ]

    for i, s in enumerate(selected_series):
        yaxis = axis_map.get(i, 'y')
        chart_type = chart_type_map.get(i, 'line')
        axis_label = "(LS)" if yaxis == 'y' else "(RS)" if yaxis == 'y2' else "(Column)"
        color = color_order[i % len(color_order)]
        invert = (yaxis == 'y' and invert_left) or (yaxis == 'y2' and invert_right)
        if chart_type == 'line':
            fig.add_trace(Scatter(
                x=data[s].index,
                y=-data[s].values if invert else data[s].values,
                mode='lines',
                name=f"{s} {axis_label}",
                yaxis=yaxis if yaxis != 'column' else 'y',
                line=dict(
                    color=color,
                    width=1.2,
                    shape='linear'
                )
            ))
        else:
            fig.add_trace(dict(
                type='bar',
                x=data[s].index,
                y=-data[s].values if invert else data[s].values,
                name=f"{s} {axis_label}",
                yaxis=yaxis if yaxis != 'column' else 'y',
                marker=dict(color=color)
            ))

    layout = Layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        showlegend=True,
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            orientation='v',
            bgcolor='rgba(0,0,0,0)',
            font=dict(
                family='Raleway SemiBold, sans-serif',
                size=8,
                color='black'
            )
        ),
        height=chart_height if chart_height else 350,
        width=chart_width if chart_width else 600,
        margin=dict(l=40, r=20, t=20, b=40),
        font=dict(family='Segoe UI, Arial', size=11, color='black'),
        xaxis=dict(
            showgrid=False,
            showline=True,
            linecolor='black',
            linewidth=1,
            ticks='inside',
            tickcolor='black',
            tickwidth=1,
            tickfont=dict(size=8, color='black'),
            mirror=True
        ),
        yaxis=dict(
            showgrid=False,
            showline=True,
            linecolor='black',
            linewidth=1,
            ticks='inside',
            tickcolor='black',
            tickwidth=1,
            tickfont=dict(size=8, color='black'),
            title=left_unit,
            mirror=True
        )
    )

    # --- Flexible axis handling ---
    # Only set manual range if both min/max are set, else let Plotly handle zoom
    if yls_min is not None and yls_max is not None:
        layout['yaxis']['range'] = [yls_min, yls_max]
        layout['yaxis']['autorange'] = False
    else:
        layout['yaxis']['autorange'] = True  # allow zoom and auto-scaling

    if axis_mode == 'dual' and 'y2' in axis_map.values():
        layout['yaxis2'] = dict(
            overlaying='y',
            side='right',
            showgrid=False,
            showline=True,
            linecolor='black',
            linewidth=1,
            ticks='inside',
            tickcolor='black',
            tickwidth=1,
            tickfont=dict(size=8, color='black'),
            title=right_unit,
            mirror=True
        )
        if yrs_min is not None and yrs_max is not None:
            layout['yaxis2']['range'] = [yrs_min, yrs_max]
            layout['yaxis2']['autorange'] = False
        else:
            layout['yaxis2']['autorange'] = True

    shapes = []
    if sr_lines:
        try:
            levels = [float(x.strip()) for x in sr_lines.split(',') if x.strip()]
            for lvl in levels:
                shapes.append(dict(
                    type='line',
                    xref='paper',
                    yref='y',
                    x0=0, x1=1,
                    y0=lvl, y1=lvl,
                    line=dict(color='black', width=0.5, dash='solid')
                ))
        except Exception:
            pass
    if 'show' in (recession_toggle or []) and 'US: RECESSION INDICATOR' in data:
        rec = data['US: RECESSION INDICATOR'].dropna()
        rec = rec[rec == 1]
        if not rec.empty:
            rec_periods = []
            start = None
            prev = None
            for date in rec.index:
                if start is None:
                    start = date
                elif (date - prev).days > 40:
                    rec_periods.append((start, prev))
                    start = date
                prev = date
            if start is not None and prev is not None:
                rec_periods.append((start, prev))
            for s, e in rec_periods:
                shapes.append(dict(
                    type='rect',
                    xref='x',
                    yref='paper',
                    x0=s,
                    x1=e,
                    y0=0,
                    y1=1,
                    fillcolor='rgba(150,150,150,0.18)',
                    line=dict(width=0),
                    layer='below'
                ))
    layout['shapes'] = shapes

    import pandas as pd
    new_range = stored_range

    if (xrange_start and xrange_start.strip()) or (xrange_end and xrange_end.strip()):
        try:
            if xrange_start and xrange_start.strip():
                start_date = pd.to_datetime(xrange_start)
            else:
                start_date = min(data[s].index[0] for s in selected_series)
            if xrange_end and xrange_end.strip():
                end_date = pd.to_datetime(xrange_end)
            else:
                end_date = max(data[s].index[-1] for s in selected_series)
            layout['xaxis']['range'] = [str(start_date.date()), str(end_date.date())]
            new_range = [str(start_date.date()), str(end_date.date())]
        except Exception:
            pass
    elif relayout_data and 'xaxis.range[0]' in relayout_data and 'xaxis.range[1]' in relayout_data:
        new_range = [relayout_data['xaxis.range[0]'], relayout_data['xaxis.range[1]']]
        layout['xaxis']['range'] = new_range
    elif stored_range:
        layout['xaxis']['range'] = stored_range

    fig.update_layout(layout)

    if footnote:
        fig.add_annotation(
            text=footnote,
            xref='paper', yref='paper',
            x=0, y=0,
            xanchor='left', yanchor='top',
            showarrow=False,
            font=dict(size=8, color='black'),
            align='left',
            yshift=-22
        )

    return fig, new_range

@app.callback(
    Output({'type': 'export-png', 'index': MATCH}, 'n_clicks'),
    Input({'type': 'export-png', 'index': MATCH}, 'n_clicks'),
    State({'type': 'chart', 'index': MATCH}, 'figure'),
    prevent_initial_call=True
)
def export_png(n_clicks, fig_data):
    if n_clicks and fig_data:
        import plotly.graph_objs as go
        import plotly.io as pio
        os.makedirs("exports", exist_ok=True)
        panel_id = ctx.triggered_id['index'] if hasattr(ctx, 'triggered_id') and ctx.triggered_id else 0
        filename = os.path.join("exports", f"chart_{panel_id}.png")
        try:
            fig = go.Figure(fig_data)
            fig.write_image(filename, scale=5)
        except Exception as e:
            print(f"Error saving PNG: {e}")
    return 0  # Reset button after click

@app.callback(
    Output({'type': 'export-svg', 'index': MATCH}, 'n_clicks'),
    Input({'type': 'export-svg', 'index': MATCH}, 'n_clicks'),
    State({'type': 'chart', 'index': MATCH}, 'figure'),
    prevent_initial_call=True
)
def export_svg(n_clicks, fig_data):
    if n_clicks and fig_data:
        import plotly.graph_objs as go
        import plotly.io as pio
        os.makedirs("exports", exist_ok=True)
        panel_id = ctx.triggered_id['index'] if hasattr(ctx, 'triggered_id') and ctx.triggered_id else 0
        filename = os.path.join("exports", f"chart_{panel_id}.svg")
        try:
            fig = go.Figure(fig_data)
            fig.write_image(filename)
        except Exception as e:
            print(f"Error saving SVG: {e}")
    return 0  # Reset button after click

if __name__ == '__main__':
    os.makedirs("exports", exist_ok=True)
    app.run(debug=True, use_reloader=False, port=8066)